# sdflow — getting started

[`sdflow`](../src/sdflow_bindings.cpp) is a Kokkos cut-cell **immersed-boundary** incompressible
Navier-Stokes solver for porous media. This notebook walks through the Python API on a small
example: a plane channel between two no-slip walls driven by a body force, compared to the analytic
Poiseuille profile.

**Conventions.** Physical units throughout (density `rho`, viscosity `mu`, physical pressure `p`);
signed-distance fields are **negative inside the solid**; 3-D fields are **Fortran-order `(nx,ny,nz)`
float64** (x is the fastest index). `sdflow.Solver` is the staggered MAC solver (the accuracy
default); `sdflow.SolverColocated` is the cell-centered variant with an identical API.

Build the module first (see [the README](../README.md) / `CMakeLists.txt`) and run with
`PYTHONPATH=build` pointing at the build that contains `sdflow*.so`.

In [ ]:
import gc
import numpy as np
import sdflow

# The active Kokkos backend is chosen by the build's install prefix (OpenMP / Cuda / HIP).
print("Kokkos backend:", sdflow.execution_space)

## 1. Define the geometry as a signed-distance field

We model a channel of height `H` (in cells) between two walls perpendicular to *y*, leaving a
couple of solid cells at each *y*-boundary. The SDF is positive in the fluid slab and negative in
the walls. It must be a Fortran-order `(nx, ny, nz)` array.

In [ ]:
nx, ny, nz = 8, 48, 8
H = ny - 4.0          # channel height; walls sit 2 cells in from each y-boundary
yc = ny / 2.0

ys = np.arange(ny) + 0.5
# fluid where |y - yc| < H/2  ->  sdf > 0 inside the channel, < 0 in the walls
Y = np.broadcast_to(ys[None, :, None], (nx, ny, nz))
sdf = np.asarray((H / 2.0) - np.abs(Y - yc), dtype=np.float64, order="F")
print("sdf shape", sdf.shape, "order F:", sdf.flags["F_CONTIGUOUS"])

## 2. Configure and run the solver

Set the physical parameters and the driving body force, hand the solver the solid SDF (enabling the
cut-cell pressure operator), then step. A large `dt` drives straight to the steady Stokes solution.

In [ ]:
rho, mu, gx = 1.0, 1.0, 1e-3

s = sdflow.Solver(nx, ny, nz)
s.set_rho(rho)
s.set_mu(mu)
s.set_dt(1e6)                       # large dt -> steady Stokes flow
s.set_body_force(gx, 0.0, 0.0)      # drive the flow along x
s.set_solid(sdf, cutcell_pressure=True)   # SDF < 0 inside the walls; no-slip on the surface

for _ in range(40):
    s.step()

print("pressure-solver iterations (last step):", s.last_pressure_iterations())
print("incompressibility residual (max cut-cell divergence):", s.max_open_divergence())

## 3. Compare to the analytic Poiseuille profile

For a body force `gx` between walls a distance `H` apart, the steady profile is
`u(y) = gx/(2 mu) * ((H/2)^2 - (y - yc)^2)`. Read the velocity back as a `(nx, ny, nz)` array and
extract the centerline column.

In [ ]:
u = s.get_u()                              # (nx, ny, nz) Fortran-order, index [x, y, z]
u_profile = u[nx // 2, :, nz // 2]

yprime = ys - yc
u_ana = np.where(np.abs(yprime) < H / 2.0,
                 gx / (2.0 * mu) * ((H / 2.0) ** 2 - yprime ** 2), 0.0)

fluid = np.abs(yprime) < H / 2.0 - 1.0
rel_err = np.abs(u_profile[fluid] - u_ana[fluid]).max() / u_ana[fluid].max()
print(f"u_max: sim={u_profile.max():.5f}  analytic={u_ana.max():.5f}  rel_err={rel_err:.2e}")

In [ ]:
# Optional plot (needs matplotlib).
try:
    import matplotlib.pyplot as plt
    plt.plot(ys, u_profile, "o", label="sdflow", ms=4)
    plt.plot(ys, u_ana, "-", label="analytic")
    plt.xlabel("y (cells)"); plt.ylabel("u"); plt.legend(); plt.title("Poiseuille channel")
    plt.show()
except ImportError:
    print("matplotlib not available — skipping the plot")

## 4. The collocated solver

`sdflow.SolverColocated` stores velocities at cell centers (vs the staggered faces) and uses the
Almgren-Bell-Colella approximate projection. Its Python API is **identical** — swap the class name.
It is ~1% less accurate per grid on integral permeability (intrinsic to the velocity placement) but
convenient for coupling to cell-centered scalars.

In [ ]:
sc = sdflow.SolverColocated(nx, ny, nz)
sc.set_rho(rho); sc.set_mu(mu); sc.set_dt(1e6)
sc.set_body_force(gx, 0.0, 0.0)
sc.set_solid(sdf, cutcell_pressure=True)
for _ in range(40):
    sc.step()
uc = sc.get_u()[nx // 2, :, nz // 2]
print(f"collocated u_max={uc.max():.5f}  (staggered {u_profile.max():.5f})")

## 5. Cleanup

The solver holds Kokkos `View`s, and Kokkos is finalized at interpreter exit via `atexit`. Release
every solver **before** exit so its device memory is freed first.

In [ ]:
del s, sc
gc.collect()